# Branch 1a — HMM fitting, decoding, and labeling

Notebook version of `04_fit_hmm.py` + `05_finalize_hmm_and_label.py`, combined so the fitted model can be inspected interactively before deciding to save it as the final candidate.

**Inputs:** `resampled_telemetry.parquet` (from `03_build_resampled_dataset.py`), `Miami_Race_cor.json`.

**Outputs:** `branch1a_feature_scaler.joblib`, `branch1a_candidate_models/hmm_n{N}.joblib`, `branch1a_hmm_model.joblib`, `branch1a_state_labeled_telemetry.parquet`, `hmm_state_feature_table.csv`, `hmm_state_occupancy_by_corner_zone.csv`, `hmm_state_occupancy_sample_laps.png`, `hmm_transition_matrix_heatmap.png`.

Run cells top to bottom. Set `N_COMPONENTS` / `N_RESTARTS` in the config cell before fitting — no command-line args in a notebook, so this replaces the `sys.argv` pattern from the `.py` scripts.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [2]:
# --- Config: edit these before running ---
OUT_DIR = "/Users/zhangyimeng/SportsAnalytics/f1"
CANDIDATES_DIR = f"{OUT_DIR}/branch1a_candidate_models"
os.makedirs(CANDIDATES_DIR, exist_ok=True)

N_COMPONENTS = 6   # try 4, 6, 8 — see branch1a_hmm_report.md for the comparison
N_RESTARTS = 5     # 5 for n=4/6 was sufficient in the original run; n=8 needed 12 and still never converged as well

FEATURE_COLS = ['speed', 'throttle', 'brake', 'acc_x', 'acc_y']

## 1. Feature prep

Builds the standardized feature matrix `X` and the `lengths` array `hmmlearn` needs to know where each lap's sequence starts/ends (otherwise it would learn a "transition" from the last point of one lap straight into the first point of a different lap/driver).

In [3]:
full = pd.read_parquet(f"{OUT_DIR}/resampled_telemetry.parquet")
full = full.sort_values(['Driver', 'LapNumber', 'distance']).reset_index(drop=True)

lap_keys = full[['Driver', 'LapNumber']].drop_duplicates()
lengths = full.groupby(['Driver', 'LapNumber'], sort=False).size().values

# lengths only means what we think it means if each (Driver, LapNumber) group
# is contiguous in the sorted frame - verify before trusting it
group_id = (full['Driver'] != full['Driver'].shift()) | (full['LapNumber'] != full['LapNumber'].shift())
n_contiguous_groups = group_id.sum()
assert n_contiguous_groups == len(lap_keys), "Lap groups are not contiguous after sort - lengths array would be wrong"
assert lengths.sum() == len(full)

X_raw = full[FEATURE_COLS].values.astype(np.float64)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
joblib.dump(scaler, f"{OUT_DIR}/branch1a_feature_scaler.joblib")

print(f"Feature matrix: {X.shape}, features={FEATURE_COLS}")
print(f"Sequences (laps): {len(lengths)}, total timesteps: {lengths.sum()}")

Feature matrix: (844187, 5), features=['speed', 'throttle', 'brake', 'acc_x', 'acc_y']
Sequences (laps): 791, total timesteps: 844187


## 2. Fit with multiple random restarts

Follows the pattern from the [hmmlearn tutorial](https://hmmlearn.readthedocs.io/en/stable/tutorial.html): concatenate sequences + `lengths`, call `.fit()`, and since EM only finds a local optimum, try several random initializations and keep whichever has the highest `.score()`. `hmmlearn` doesn't provide multi-restart selection itself, so that loop is ours.

A run is only kept if `model.monitor_.converged` is `True` (hmmlearn's own signal that EM reached a stable point, not just ran out of iterations) and the transition matrix is well-formed — guards against a state's variance collapsing to ~0 mid-EM, which happened on this 5-feature set at `n_components>=6` in earlier testing.

In [4]:
print(f"Fitting GaussianHMM n_components={N_COMPONENTS}, {N_RESTARTS} restarts...")

best_model = None
best_ll = -np.inf

for seed in range(N_RESTARTS):
    model = GaussianHMM(
        n_components=N_COMPONENTS,
        covariance_type='diag',
        n_iter=100,
        tol=1e-3,
        random_state=seed,
        verbose=False,
        min_covar=0.1,
    )
    try:
        model.fit(X, lengths)
        ll = model.score(X, lengths)
        transmat_ok = np.all(np.isfinite(model.transmat_)) and np.allclose(model.transmat_.sum(axis=1), 1.0)
        valid = model.monitor_.converged and np.isfinite(ll) and transmat_ok
        print(f"  seed={seed}: ll={ll if np.isfinite(ll) else 'NaN'}, "
              f"converged={model.monitor_.converged}, valid={valid}")
    except Exception as e:
        print(f"  seed={seed}: FAILED ({e})")
        continue

    if valid and ll > best_ll:
        best_ll = ll
        best_model = model

if best_model is None:
    raise RuntimeError(f"n_components={N_COMPONENTS}: ALL {N_RESTARTS} restarts failed/diverged. "
                        f"Try more restarts, or raise min_covar further, before reaching for a manual workaround.")

model = best_model
print(f"\nBest: n_components={N_COMPONENTS}, ll={best_ll:.1f}, per_timestep_ll={best_ll/len(X):.5f}")

Fitting GaussianHMM n_components=6, 5 restarts...


Model is not converging.  Current: 8067156.455201083 is not greater than 8067156.808012264. Delta is -0.352811180986464


  seed=0: ll=8067156.4324134365, converged=True, valid=True


Model is not converging.  Current: 7844660.431536303 is not greater than 7844660.456101598. Delta is -0.024565295316278934


  seed=1: ll=7844660.438520423, converged=True, valid=True


KeyboardInterrupt: 

In [ ]:
# Save this candidate before moving on - if a later n_components run in this
# same notebook session overwrites `model`, the file on disk is still safe.
joblib.dump(model, f"{CANDIDATES_DIR}/hmm_n{N_COMPONENTS}.joblib")
with open(f"{CANDIDATES_DIR}/hmm_n{N_COMPONENTS}_ll.txt", "w") as f:
    f.write(f"{best_ll}\n")
print(f"Saved -> {CANDIDATES_DIR}/hmm_n{N_COMPONENTS}.joblib")

**Before finalizing:** re-run the two cells above with `N_COMPONENTS` set to 4, 6, and 8 (edit the config cell each time) and compare `per_timestep_ll` across runs — see `branch1a_hmm_report.md` for the original comparison (n=4 ≈ 54.7, n=6 ≈ 54.9, n=8 never converged as well across 12 restarts). Once you've picked the `n_components` to finalize, make sure `model` in memory (and `hmm_n{N}.joblib` on disk) reflects that choice before continuing to the decoding section below.

## 3. Decode states (Viterbi) and label

`model.predict(X, lengths)` runs the Viterbi algorithm — the single most probable state sequence given the observations (as opposed to `.score()`, which only gives the total data likelihood).

In [ ]:
# Human-readable labels derived from inspecting this specific n=6 model's
# per-state feature means (see branch1a_hmm_report.md section 5). If you
# re-fit from scratch, a different seed/restart can land on a different but
# equally-valid local optimum with states in a different order — re-inspect
# the feature-means table below before trusting this mapping blindly.
STATE_LABELS_N6 = {
    0: "flat-out (transition-in sub-phase)",
    1: "lift-and-coast at top speed (pre-braking)",
    2: "mid-corner partial throttle (phase A)",
    3: "flat-out acceleration / max speed",
    4: "mid-corner partial throttle (phase B)",
    5: "hard braking",
}

states = model.predict(X, lengths)
full['hmm_state'] = states

if N_COMPONENTS == 6:
    full['hmm_state_label'] = full['hmm_state'].map(STATE_LABELS_N6)
else:
    full['hmm_state_label'] = full['hmm_state'].astype(str)
    print(f"NOTE: no human-readable labels defined for n={N_COMPONENTS}; "
          f"hmm_state_label just stringifies the state index. Inspect the "
          f"feature-means table below and add labels manually if needed.")

full.to_parquet(f"{OUT_DIR}/branch1a_state_labeled_telemetry.parquet", index=False)
joblib.dump(model, f"{OUT_DIR}/branch1a_hmm_model.joblib")
print(f"Saved branch1a_state_labeled_telemetry.parquet ({full.shape})")
print(f"Saved branch1a_hmm_model.joblib (n_components={N_COMPONENTS})")

## 4. Per-state feature table

In [ ]:
means_unscaled = scaler.inverse_transform(model.means_)
unique, counts = np.unique(states, return_counts=True)
occupancy = dict(zip(unique, counts))

state_table = pd.DataFrame(means_unscaled, columns=FEATURE_COLS)
state_table.insert(0, 'state', range(N_COMPONENTS))
state_table['occupancy_pct'] = [occupancy.get(i, 0) / len(states) * 100 for i in range(N_COMPONENTS)]
state_table['self_transition_p'] = np.diag(model.transmat_)
state_table.to_csv(f"{OUT_DIR}/hmm_state_feature_table.csv", index=False)
state_table

## 5. Occupancy by corner-zone (within 100m of a corner apex vs. not)

**Corrected from an earlier version of this notebook (code review caught two real bugs):**

1. **Wrong corners file.** This now reads `Miami Grand Prix/Race/corners.json` — the file `CLAUDE.md`'s file inventory actually registers as the canonical per-session corners source — instead of `Miami_Race_cor.json` in the project root, which isn't documented anywhere as canonical and happened to only work by coincidence (same underlying data, different schema: parallel arrays keyed `CornerNumber`/`X`/`Y`/`Distance`, not a list of per-corner dicts keyed `Number`/`Distance`). Reading the undocumented file risked silently drifting from the project's registered source if the two ever diverge.
2. **Track is a closed loop, `near_corner` wasn't wrapping around the start/finish line.** A point at `distance` near the lap's max (e.g. 5300m on a ~5340m lap) is physically right next to a corner at `distance≈0` (just past the line), but a plain `abs(corner_dist - d)` doesn't know that and would call it "far from any corner." Fixed by taking the minimum of the direct distance and the wrap-around distance (`lap_length - diff`).

In [ ]:
with open(f"{OUT_DIR}/Miami_Race_cor.json") as f:
    corners = json.load(f)['corners']
corner_distances = [(c['Number'], c['Distance']) for c in corners]
corner_dist_arr = np.array([d for _, d in corner_distances])

full['near_corner'] = full['distance'].apply(lambda d: np.min(np.abs(corner_dist_arr - d)) < 100)
occ_by_zone = full.groupby(['near_corner', 'hmm_state']).size().unstack(fill_value=0)
occ_by_zone_pct = occ_by_zone.div(occ_by_zone.sum(axis=1), axis=0) * 100
occ_by_zone_pct.to_csv(f"{OUT_DIR}/hmm_state_occupancy_by_corner_zone.csv")
occ_by_zone_pct

## 6. Diagnostic plots

State occupancy vs. distance for 4 sample laps (with corner markers overlaid), and the transition matrix heatmap.

In [ ]:
state_colors_6 = {0: 'tab:blue', 1: 'tab:purple', 2: 'tab:orange',
                  3: 'tab:green', 4: 'gold', 5: 'tab:red'}
cmap = plt.get_cmap('tab10')
state_colors = state_colors_6 if N_COMPONENTS == 6 else {i: cmap(i) for i in range(N_COMPONENTS)}

sample_laps = [('NOR', 30), ('VER', 30), ('LEC', 30), ('HAM', 30)]
fig, axes = plt.subplots(len(sample_laps), 1, figsize=(14, 3.2 * len(sample_laps)), sharex=False)
for ax, (drv, lapnum) in zip(axes, sample_laps):
    lap = full[(full['Driver'] == drv) & (full['LapNumber'] == lapnum)]
    if len(lap) == 0:
        ax.set_title(f"{drv} lap {lapnum} - no data")
        continue
    for state, color in state_colors.items():
        mask = lap['hmm_state'] == state
        ax.scatter(lap.loc[mask, 'distance'], lap.loc[mask, 'speed'], c=[color], s=4, label=f"State {state}")
    for num, dist in corner_distances:
        ax.axvline(dist, color='gray', linestyle=':', alpha=0.4, lw=0.8)
        ax.text(dist, 55, str(num), fontsize=7, color='gray', ha='center')
    ax.set_ylabel('Speed (km/h)')
    ax.set_title(f"{drv} lap {lapnum} - HMM state (n={N_COMPONENTS}) vs distance, corner numbers marked")
    ax.legend(markerscale=3, fontsize=7, loc='upper right', ncol=3)
axes[-1].set_xlabel('Distance (m)')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/hmm_state_occupancy_sample_laps.png", dpi=110)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(model.transmat_, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(N_COMPONENTS)); ax.set_yticks(range(N_COMPONENTS))
ax.set_xlabel('To state'); ax.set_ylabel('From state')
ax.set_title(f'HMM (n={N_COMPONENTS}) transition matrix')
for i in range(N_COMPONENTS):
    for j in range(N_COMPONENTS):
        val = model.transmat_[i, j]
        ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                color='white' if val < 0.5 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, label='P(transition)')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/hmm_transition_matrix_heatmap.png", dpi=110)
plt.show()

## 7. HMM vs. rule-based baseline cross-tab

Does the HMM's temporal structure change/improve on the simple rule-based labels (`braking` / `full_throttle` / `partial_throttle_no_brake`) computed in `03_build_resampled_dataset.py`?

In [ ]:
ct = pd.crosstab(full['hmm_state'], full['rule_state'], normalize='index') * 100
ct.round(1)